In [1]:
!pip uninstall -y qiskit qiskit-aer qiskit-ibm-runtime

!pip install -q \
    "qiskit==2.3.0" \
    "qiskit-aer==0.17.2" \
    "qiskit-ibm-runtime==0.45.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 4.5 MB/s eta 0:00:00


In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from getpass import getpass

api_key = getpass("Paste your IBM Cloud API key here: ")

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=api_key
)

print("Connected to IBM Quantum")

Paste your IBM Cloud API key here: ··········


qiskit_runtime_service._discover_account:WARNING:2026-09-15 02:40:24,225: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-09-15 02:40:26,950: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


Connected to IBM Quantum


In [3]:
from datetime import datetime, timezone

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.transpiler import generate_preset_pass_manager


# ------------------------------------------------------------
# 1. LOAD IBM FEZ AND THE HISTORICAL AUGUST TARGET
# ------------------------------------------------------------

backend = service.backend("ibm_fez")

target_time = datetime(
    2026, 8, 24, 3, 20, 0,
    tzinfo=timezone.utc
)

historical_properties = backend.properties(
    datetime=target_time
)

historical_target = backend.target_history(
    datetime=target_time
)

print(
    "Calibration timestamp:",
    historical_properties.last_update_date
)


# ------------------------------------------------------------
# 2. RECREATE THE SAME 4-QUBIT HQNN CIRCUIT
# ------------------------------------------------------------

x = ParameterVector("x", 4)
w = ParameterVector("w", 24)

reference_circuit = QuantumCircuit(4)

# Angle embedding
for qubit in range(4):
    reference_circuit.ry(
        x[qubit],
        qubit
    )

parameter_index = 0

# First quantum layer
for qubit in range(4):

    reference_circuit.rz(
        w[parameter_index],
        qubit
    )
    parameter_index += 1

    reference_circuit.ry(
        w[parameter_index],
        qubit
    )
    parameter_index += 1

    reference_circuit.rz(
        w[parameter_index],
        qubit
    )
    parameter_index += 1


# First entangling layer
for control in range(4):

    target = (control + 1) % 4

    reference_circuit.cx(
        control,
        target
    )


# Second quantum layer
for qubit in range(4):

    reference_circuit.rz(
        w[parameter_index],
        qubit
    )
    parameter_index += 1

    reference_circuit.ry(
        w[parameter_index],
        qubit
    )
    parameter_index += 1

    reference_circuit.rz(
        w[parameter_index],
        qubit
    )
    parameter_index += 1


# Second entangling layer
for control in range(4):

    target = (control + 2) % 4

    reference_circuit.cx(
        control,
        target
    )


# ------------------------------------------------------------
# 3. TRANSPILE USING THE SAME PHYSICAL QUBITS
# ------------------------------------------------------------

pass_manager = generate_preset_pass_manager(
    target=historical_target,
    optimization_level=3,
    initial_layout=[33, 34, 35, 39],
    seed_transpiler=42,
    scheduling_method="asap"
)

transpiled_circuit = pass_manager.run(
    reference_circuit
)


# ------------------------------------------------------------
# 4. CALCULATE PHYSICAL CIRCUIT DURATION
# ------------------------------------------------------------

duration_us = transpiled_circuit.estimate_duration(
    historical_target,
    unit="u"
)

print(
    "Circuit depth:",
    transpiled_circuit.depth()
)

print(
    "Circuit duration:",
    duration_us,
    "microseconds"
)


# ------------------------------------------------------------
# 5. COMPARE WITH COHERENCE TIMES
# ------------------------------------------------------------

min_T1 = 94.71
min_T2 = 37.58

print(
    "Shortest T1:",
    min_T1,
    "microseconds"
)

print(
    "Shortest T2:",
    min_T2,
    "microseconds"
)

print(
    "Circuit / T1:",
    duration_us / min_T1
)

print(
    "Circuit / T2:",
    duration_us / min_T2
)

qiskit_runtime_service.backends:WARNING:2026-09-15 02:40:51,402: Using instance: open-instance, plan: open


Calibration timestamp: 2026-08-24 03:06:47+00:00
Circuit depth: 54
Circuit duration: 1.2960000000000003 microseconds
Shortest T1: 94.71 microseconds
Shortest T2: 37.58 microseconds
Circuit / T1: 0.013683877098511248
Circuit / T2: 0.034486428951569996
